In [1]:
import numpy as np

$\chi ^2( \theta_k) = (y_i - f(x_i, \theta_k ))C_{ij}^{-1}(y_j - f(x_j, \theta_k))$

In [210]:
# Definimos funciones f, chi^2 y una propuesta diagonal de C

# Para este caso f sera la ecuacion de la recta, pero puede cambiarse a cualquier otra funcion a la que se desee ajustar los datos
def f(x, theta):
    return theta[0]*x + theta[1] #theta[0] sera la pendiente y theta[1] la coordenada de origen

# Definimos una propuesta de C, donde todos los errores son iguales (1) y no estan correlacionados, es decir la matriz C es lineal
def defC(y):
    return np.eye(len(y))

# Ahora definimos a la funcion chi^2, para practicidad, haremos primero una funcion que calcule la inversa de C
def Cinv(C):
    return np.linalg.inv(C)

# Para optimizar el calculo de la minimizacion se calculara chi^2 no solo para los parametros de preba, al mismo tiempo se calculara para los parametros posibles al rededor
# Hacer esto requiere que tengamos el vector de datos en la dimensionalidad adecuada y de f en evaluada en los parametros de alrededor y el central
def dimenfixdata(y):
    return np.broadcast_to(y, (3, 3, 100))

def dimenfixfunc(x, theta, esp):
    param = np.array([theta-esp, theta, theta+esp])
    func = np.array([[f(x, [param[0][0],param[0][1]]), f(x, [param[0][0],param[1][1]]), f(x, [param[0][0],param[2][1]])],
                     [f(x, [param[1][0],param[0][1]]), f(x, [param[1][0],param[1][1]]), f(x, [param[1][0],param[2][1]])],
                     [f(x, [param[2][0],param[0][1]]), f(x, [param[2][0],param[1][1]]), f(x, [param[2][0],param[2][1]])]])
    return func
    
def chi2(y, x, Cinv, theta, esp):
    f = dimenfixfunc(x, theta, esp)
    vec = y-f
    c2 = np.einsum('ijk,kl,ijl->ij', vec, Cinv, vec)
    return c2 #Se calcula de forma matricial pues asi es mucho mas rapido que usando ciclos for

In [211]:
theta = np.array([-0.3, 0.2])

esp = np.array([0.0001, 0.0001])

x = np.linspace(0,10, 100)

# generamos datos para hacer el ajuste lineal
y = np.random.normal(3*x+1, 0.5)
ymat = dimenfixdata(y)

C_inv = Cinv(defC(y))

In [212]:
limit = 0.000001
diff = 1
c=0

while (diff>=limit and c<100000):
    c2 = chi2(y, x, C_inv, theta, esp) #Se calcula chi^2 y sus alrededores
    i,j=np.unravel_index(np.argmin(c2), c2.shape)
    param = np.array([theta-esp, theta, theta+esp])
    theta = [param[i][0], param[j][1]] #se guarda la combinacion de parametros que mejor ajusta los datos

    # A continuacion se calcular la diferencia entre los alrededores y el central, si son muy parecidos (limit), para la minimizacion
    centro = c2[1, 1]

    # máscara para excluir el centro
    mask = np.ones((3, 3), dtype=bool)
    mask[1, 1] = False
    
    # vecinos
    vecinos = c2[mask]
    
    # diferencias y promedio
    diff = np.mean(vecinos - centro)
    c+=1

In [213]:
theta

[np.float64(2.9868000000018617), np.float64(1.065399999999905)]